# 2. Evaluate and release (GPU)

Runs the benchmark (base vs base+3 examples vs adapted), writes the model and dataset cards, and publishes on Hugging Face and Kaggle. Run the parts in order; stop after part C to read the results first.

**Settings:** Accelerator = **GPU T4 x2**, Internet = **On** (phone-verified Kaggle account).

**Add-ons → Secrets** (tick each for this notebook): `HF_TOKEN` (Hugging Face **write** token), `ADAPTION_API_KEY` (only for part B), `KAGGLE_USERNAME` and `KAGGLE_KEY` (from your kaggle.json, for part E).

## A. Settings

In [ ]:
REPO = 'https://github.com/YOUR_GITHUB_USER/nema-agro-adaptive.git'
PROJECT = 'agro'                         # 'agro' or 'waste'
HF_USER = 'YOUR_HF_USER'
KAGGLE_USER = 'YOUR_KAGGLE_USER'
ADAPTER = f'{HF_USER}/nema-agro-lora' if PROJECT == 'agro' else f'{HF_USER}/revalo-waste-lora'
ADAPTION_RUN_ID = ''   # AutoScientist run id: fill it to let part B copy the weights to Hugging Face for you
BASE = 'Qwen/Qwen3.5-9B'  # the exact base model you chose in AutoScientist (official Hugging Face id)
PER_BUCKET = 40          # test examples per task and split; 25 if your GPU quota is short

import os
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
for name in ('HF_TOKEN', 'ADAPTION_API_KEY', 'KAGGLE_USERNAME', 'KAGGLE_KEY'):
    try:
        os.environ[name] = secrets.get_secret(name)
    except Exception:
        print('secret not set (fine if you skip the part that needs it):', name)

In [ ]:
!git clone -q $REPO kit
%cd kit
!pip install -q -U transformers peft bitsandbytes accelerate huggingface_hub kagglehub adaption
!bash scripts/build_all.sh > /dev/null && echo 'datasets rebuilt'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# The rebuilt test set must be the one frozen by notebook 1 (committed to GitHub), or results are not comparable.
import subprocess
test_file = f'data/build/{PROJECT}/test.jsonl'
if subprocess.run(['git', 'ls-files', '--error-unmatch', test_file], capture_output=True).returncode:
    print(f'WARNING: {test_file} is not committed, so this run cannot prove it uses the same test set as notebook 1.')
    print('Commit the test.jsonl from notebook 1 (see its last cell) and run again.')
else:
    changed = subprocess.run(['git', 'diff', '--quiet', '--', test_file]).returncode
    assert not changed, (f'the rebuilt {test_file} differs from the committed one: the lexicon, generators or seeds '
                         'changed after notebook 1. Run notebook 1 again and commit its test.jsonl.')
    print('test set matches the committed frozen copy')

## B. Copy the trained weights from Adaption to Hugging Face (skip if already there)

No-code alternative: in the Adaption app, open the trained model, click **Weights export** to download the archive, create an empty model repo on huggingface.co and upload the extracted files with **Add file → Upload files**.

In [ ]:
import glob, tarfile
from pathlib import Path
from adaption import Adaption
from huggingface_hub import HfApi

assert ADAPTION_RUN_ID, 'set ADAPTION_RUN_ID in part A'
client = Adaption()  # reads ADAPTION_API_KEY
run = client.autoscientist.get(ADAPTION_RUN_ID)
print('status =', run.status, '| download_available =', getattr(run, 'download_available', None))
assert getattr(run, 'download_available', False), 'run not finished or no download yet: use the no-code path above'
# streamed download of the best checkpoint (see docs: AutoScientist > Download the model)
with client.autoscientist.with_streaming_response.download(ADAPTION_RUN_ID) as resp:
    resp.stream_to_file('best-checkpoint.tgz')
Path('best-checkpoint').mkdir(exist_ok=True)
with tarfile.open('best-checkpoint.tgz') as t:
    t.extractall('best-checkpoint')
adapter_dir = str(Path(glob.glob('best-checkpoint/**/adapter_config.json', recursive=True)[0]).parent)
print('adapter files:', os.listdir(adapter_dir))
api = HfApi(token=os.environ['HF_TOKEN'])
api.create_repo(ADAPTER, repo_type='model', exist_ok=True, private=True)
api.upload_folder(folder_path=adapter_dir, repo_id=ADAPTER, commit_message='LoRA adapter from Adaption AutoScientist')
print('uploaded to https://huggingface.co/' + ADAPTER, '(private until part E)')

## C. Benchmark: base, base + 3 examples, adapted

In [ ]:
!python -m src.eval.run_eval --test data/build/$PROJECT/test.jsonl --adapter $ADAPTER --base $BASE --out results/$PROJECT --per-bucket $PER_BUCKET --batch-size 8 --fewshot 3
!python -m src.eval.score --pred results/{PROJECT}_base.jsonl results/{PROJECT}_base3shot.jsonl results/{PROJECT}_adapted.jsonl --out results/{PROJECT}

In [ ]:
from IPython.display import Markdown
Markdown(open(f'results/{PROJECT}_results.md').read())

Read 10 outputs per task in `results/<project>_adapted.jsonl` before trusting the numbers. Stop here if something looks wrong.

## D. Model and dataset cards

Fill the values below (from AutoScientist's results page and your Adaptive Data runs), then run.

In [ ]:
import json
cfg_path = f'configs/project_{PROJECT}.json'
cfg = json.load(open(cfg_path))
cfg.update({
    'hf_user': HF_USER, 'kaggle_user': KAGGLE_USER, 'base_model': BASE,
    'hf_model_repo': ADAPTER,
    'hf_dataset_repo': f"{HF_USER}/{'nema-agro' if PROJECT == 'agro' else 'revalo-waste'}-dataset",
    'kaggle_model_handle': f"{KAGGLE_USER}/{'nema-agro' if PROJECT == 'agro' else 'revalo-waste'}/transformers/lora-v1",
    'kaggle_dataset_handle': f"{KAGGLE_USER}/{'nema-agro' if PROJECT == 'agro' else 'revalo-waste'}-dataset",
    'github_repo': REPO.removesuffix('.git'),
    'kaggle_notebook_url': 'PASTE the public URL of this notebook',
    'eval_time': 'PASTE how long part C took',
    'autoscientist_winrate': 'PASTE from AutoScientist > Training Winrates (e.g. 68%)',
})
# training_steps and sources: edit configs/project_<project>.json in GitHub, or change cfg['training_steps'] here
json.dump(cfg, open(cfg_path, 'w'), ensure_ascii=False, indent=2)
!python -m src.make_cards --project $PROJECT
!grep -n 'REPLACE\|PASTE' release/$PROJECT/model/README.md release/$PROJECT/dataset/README.md || echo 'cards complete'

## E. Publish on Hugging Face and Kaggle

Runs only when no `REPLACE` is left. Then make the Hugging Face repos public (repo → Settings → Change visibility) and finish the Kaggle pages in the browser (playbook section 10).

In [ ]:
!python -m src.publish hf-dataset    --project $PROJECT
!python -m src.publish hf-model-card --project $PROJECT
!python -m src.publish kaggle-model  --project $PROJECT
!python -m src.publish kaggle-dataset --project $PROJECT

In [ ]:
# keep results and cards in the notebook Output (Save Version -> Save & Run All)
import shutil
shutil.copytree('results', '/kaggle/working/results', dirs_exist_ok=True)
shutil.copytree('release', '/kaggle/working/release', dirs_exist_ok=True)

## F. Optional: live demo with a temporary public link
Runs the Gradio demo in 4-bit on this GPU. The `*.gradio.live` link works while the cell runs.

In [ ]:
!pip install -q gradio
!cd demo && LOAD_4BIT=1 GRADIO_SHARE=1 BASE_MODEL={BASE} AGRO_ADAPTER={ADAPTER} python app.py